# 2 — Grid Search Configurations

Finds the model and vectorizer settings that predict the **full Level 1–4
path** most accurately, scored by k-fold cross-validation.

Specify only the parameters you want to vary — anything left out keeps the
value already in the config. The winning settings are written to JSON so
notebooks 3 and 4 pick them up without anything being retyped.

Runtime is `candidates × folds` model fits, which grows fast. Use
`SAMPLE_FRAC` for a wide first pass, then confirm the shortlist on
everything.

## Setup

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import json
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd

from product_classifier import PipelineConfig, data as data_mod, paths
from product_classifier.gridsearch import (
    grid_search, searchable_parameters, print_parameter_reference, summarise_by,
    best_params, apply_params, save_params, load_params,
)
from product_classifier.tradeoffs import estimate_memory

---
## Config

Only the data settings matter here. Depth is fixed at full by the search,
and anything in the grid overrides what is set below.

In [0]:
cfg = PipelineConfig()

cfg.data.path = paths.data_path("cleaned_data.txt")
cfg.data.file_format = "csv"
cfg.data.delimiter = "|"
cfg.data.text_col = "FullDesc"
cfg.data.id_col = "Vallen ID"

# IDs rather than names: joint_path matches labels by string equality, and
# a name can repeat under different parents.
cfg.data.level_columns = [
    "Level 1 ID",
    "Level 2 ID",
    "Level 3 ID",
    "Level 4 ID",
]

cfg.split.min_path_count = 20
cfg.model.fit_chunk_size = 20000    # bounds memory; does not change results
cfg.model.batch_size = 1000

levels = cfg.active_levels()
df = data_mod.load_data(cfg.data)
print(f"{len(df):,} rows | predicting {levels}")

603,421 rows | predicting ['Level 1 ID', 'Level 2 ID', 'Level 3 ID', 'Level 4 ID']


### What can be searched

Every searchable parameter, with **its default**, the value your config
currently holds, whether they differ, what the parameter does, and values
worth trying. A path not on this list raises rather than being silently
ignored, so a typo costs nothing but a moment.

`split.min_path_count` is the **rare-path threshold** — categories with
fewer products than this are dropped before training.

In [0]:
searchable_parameters(cfg)

,parameter,default,current,changed,example_values,what_it_does
0,model.vectorizer.kind,'tfidf','tfidf',False,"'tfidf', 'count'",tfidf weights rare words up; count uses raw fr...
1,model.vectorizer.max_features,40000,40000,False,"20000, 50000, 100000, None",vocabulary cap. Bigger reads more descriptions...
2,model.vectorizer.ngram_range,"(1, 2)","(1, 2)",False,"(1, 1), (1, 2), (1, 3)","(1,1) single words; (1,2) adds adjacent pairs,..."
3,model.vectorizer.min_df,2,2,False,"1, 2, 5",drop words appearing in fewer than this many d...
4,model.vectorizer.max_df,0.95,0.95,False,"0.9, 0.95, 1.0",drop words appearing in more than this share -...
5,model.vectorizer.stop_words,'english','english',False,"'english', None",'english' removes common function words; None ...
6,model.vectorizer.sublinear_tf,True,True,False,"True, False",dampen repeated words with 1+log(tf); usually ...
7,model.vectorizer.binary,False,False,False,"True, False",presence rather than count. count vectorizer only
8,model.vectorizer.norm,'l2','l2',False,"'l2', 'l1', None","row normalisation, so long descriptions do not..."
9,model.vectorizer.use_idf,True,True,False,"True, False",weight by inverse document frequency. tfidf only


In [0]:
# The same thing as text, since the notes column truncates in a DataFrame.
print_parameter_reference(cfg)


model.vectorizer.kind
    default : 'tfidf'
    tfidf weights rare words up; count uses raw frequencies
    try     : 'tfidf', 'count'

model.vectorizer.max_features
    default : 40000
    vocabulary cap. Bigger reads more descriptions but costs memory linearly, and min_df often binds first
    try     : 20000, 50000, 100000, None

model.vectorizer.ngram_range
    default : (1, 2)
    (1,1) single words; (1,2) adds adjacent pairs, which catches 'ball valve' as a unit
    try     : (1, 1), (1, 2), (1, 3)

model.vectorizer.min_df
    default : 2
    drop words appearing in fewer than this many descriptions. 1 keeps every typo and part number
    try     : 1, 2, 5

model.vectorizer.max_df
    default : 0.95
    drop words appearing in more than this share -- they carry no discriminating information
    try     : 0.9, 0.95, 1.0

model.vectorizer.stop_words
    default : 'english'
    'english' removes common function words; None keeps them
    try     : 'english', None

model.vectorizer.

### The grid

As few or as many parameters as you like — one entry is a valid search,
and an empty dict just scores the config as it stands.

Start wide and cheap. Whatever turns out not to matter can be dropped from
the next pass, which is what keeps the combination count manageable.

In [0]:
GRID = {
    "model.nb.alpha": [0.001],  # found through experimentation
    # "model.vectorizer.max_features": [60_000],  # found through experimentation

    "model.vectorizer.max_features": [200_000, 150_000, 100_000],

    "split.min_path_count": [20],  # found through experimentation
    "model.vectorizer.ngram_range": [(1,2)], # found through experimentation
    # "model.vectorizer.min_df": [1], # found through experimentation

    "model.vectorizer.min_df": [2], # to see if longer vectors can work without exceeding memory

    "model.vectorizer.max_df": [0.95], # found through experimentation
    "model.vectorizer.stop_words": ["english"],  # found through experimentation
    "model.vectorizer.kind": ["tfidf"], # found through experimentation
    "model.vectorizer.sublinear_tf": [True], # found through experimentation
    "model.vectorizer.norm": ["l2"], # , "l1", None],  # found through experimentation
    "model.vectorizer.use_idf": [False], # found through experimentation
    # "model.vectorizer.binary": [True, False], # will be binary by default since descs don't repeat words
    "model.nb.fit_prior": [True], # found through experimentation
    "model.prediction_mode": ["joint_path"], # , "per_level"], # found through experimentation
    "model.vectorizer.unordered_bigrams": [False] # found through experimentation, True],
}

N_FOLDS = 3            # 3 to explore, 5 to confirm
SAMPLE_FRAC = 0.5     # None to use every row

_n = int(np.prod([len(v) for v in GRID.values()])) if GRID else 1
print(f"{_n} candidates x {N_FOLDS} folds = {_n * N_FOLDS} model fits")

3 candidates x 3 folds = 9 model fits


**Searching `split.min_path_count` is different from the rest.** Every
other parameter changes how the model reads the same products; this one
changes *which products exist*. Each candidate is therefore scored on a
different row set, and the accuracies are not strictly comparable — a
higher threshold looks better partly because it has removed the hard
categories.

The results table reports `n_rows`, `n_paths` and
`min_path_count_applied` for exactly this reason. Read them alongside the
accuracy: a candidate that wins while dropping 20% of the catalogue has
not necessarily won.

Values below `N_FOLDS` are raised to it — a stratified split needs at
least that many products per category.

### Sampling, for efficiency

`SAMPLE_FRAC` samples stratified on the deepest level, so every category
keeps a proportional share rather than the rare ones vanishing. Two things
follow from using it:

- Accuracy comes out **lower** than a full run — less training data — so
  treat the numbers as a *ranking*, not an estimate.
- It thins the rare tail, so paths near `min_path_count` may drop out
  entirely and the surviving taxonomy is slightly easier.

Both are fine for choosing between candidates. Neither is fine for
quoting a figure, which is what notebook 3 is for.

### Memory check

The largest `max_features` in the grid combined with the lowest
`min_path_count` is the worst case, and it runs somewhere in the middle of
the sweep rather than at the end. Cheaper to know now.

In [0]:
_feature_values = GRID.get("model.vectorizer.max_features",
                           [cfg.model.vectorizer.max_features])
_path_values = GRID.get("split.min_path_count", [cfg.split.min_path_count])

_rows = []
for _f in _feature_values:
    if _f is None:
        continue
    _probe = apply_params(cfg, {"model.vectorizer.max_features": _f})
    _est = estimate_memory(_probe, df, thresholds=tuple(_path_values),
                           sample_frac=SAMPLE_FRAC)
    for _, _r in _est[_est["usable"]].iterrows():
        _rows.append({"max_features": _f,
                      "min_path_count": _r["min_path_count"],
                      "classes_deepest": _r["classes_deepest"],
                      "projected_peak_gb": round(_r["projected_peak_gb"], 2)})

pd.DataFrame(_rows).sort_values("projected_peak_gb", ascending=False)

,max_features,min_path_count,classes_deepest,projected_peak_gb
0,200000,20,1658,7.61
1,150000,20,1658,5.78
2,100000,20,1658,3.94


### Run

In [0]:
results = grid_search(
    cfg, df, GRID,
    n_folds=N_FOLDS,
    sample_frac=SAMPLE_FRAC,
)
results

[grid] 3 candidate(s) x 3 folds = 9 fits re-filtered per candidate, scoring all 4 levels
[grid] searching split.min_path_count: each candidate is scored on a different row set, so accuracies are not strictly comparable. Compare n_rows alongside.
[fit] vectorized 189,805 rows into 79,714 features (sparse, nnz=1,254,739) in 2.2s
[fit]   chunked fit: 189,805 rows in 10 chunk(s) of 20,000, 23 classes
[fit] trained 'Level 1 ID': 23 classes in 0.5s
[fit]   chunked fit: 189,805 rows in 10 chunk(s) of 20,000, 129 classes
[fit] trained 'Level 2 ID': 129 classes in 2.5s
[fit]   chunked fit: 189,805 rows in 10 chunk(s) of 20,000, 486 classes
[fit] trained 'Level 3 ID': 486 classes in 9.7s
[fit]   chunked fit: 189,805 rows in 10 chunk(s) of 20,000, 1,658 classes
[fit] trained 'Level 4 ID': 1,658 classes in 60.1s
[fit] 1,658 paths observed in training vs. 2,390,772,996 possible combinations -> scoring 0.000069% of the cross product
[fit] vectorized 189,805 rows into 79,749 features (sparse, nnz=1,2

,model.nb.alpha,model.vectorizer.max_features,split.min_path_count,model.vectorizer.ngram_range,model.vectorizer.min_df,model.vectorizer.max_df,model.vectorizer.stop_words,model.vectorizer.kind,model.vectorizer.sublinear_tf,model.vectorizer.norm,model.vectorizer.use_idf,model.nb.fit_prior,model.prediction_mode,model.vectorizer.unordered_bigrams,n_rows,n_paths,min_path_count_applied,path_accuracy,path_accuracy_std,correct_through_Level 1 ID,correct_through_Level 2 ID,correct_through_Level 3 ID,correct_through_Level 4 ID,fit_seconds
0,0.001,200000,20,"(1, 2)",2,0.95,english,tfidf,True,l2,False,True,joint_path,False,284708,1658,20,0.733074,0.001406,0.907653,0.878486,0.824571,0.733074,216.337324
1,0.001,150000,20,"(1, 2)",2,0.95,english,tfidf,True,l2,False,True,joint_path,False,284708,1658,20,0.733074,0.001406,0.907653,0.878486,0.824571,0.733074,223.203180
2,0.001,100000,20,"(1, 2)",2,0.95,english,tfidf,True,l2,False,True,joint_path,False,284708,1658,20,0.733074,0.001406,0.907653,0.878486,0.824571,0.733074,208.359770


### Read the spread before believing the winner

If the gap between first and second place is smaller than
`path_accuracy_std`, the ordering is noise — either candidate would do, so
prefer the cheaper or simpler one. `fit_seconds` shows what each setting
costs as well as what it buys.

In [0]:
_cols = ["path_accuracy", "path_accuracy_std", "fit_seconds"]
if "n_rows" in results.columns:
    _cols = ["n_rows", "n_paths", "min_path_count_applied"] + _cols
_cols += [c for c in results.columns if c.startswith("correct_through_")]

_top = results.head(10)[list(GRID) + _cols].copy()
if len(results) > 1:
    _gap = results["path_accuracy"].iloc[0] - results["path_accuracy"].iloc[1]
    _std = results["path_accuracy_std"].iloc[0]
    print(f"gap to second place: {_gap:.4f} | winner's fold spread: {_std:.4f}")
    print("-> the ordering is "
          + ("within noise; prefer the simpler candidate" if _gap < _std
             else "larger than the fold spread"))
if "n_rows" in results.columns and results["n_rows"].nunique() > 1:
    print(f"-> row counts vary from {results['n_rows'].min():,} to "
          f"{results['n_rows'].max():,}; accuracies are not directly comparable")
_top

gap to second place: 0.0000 | winner's fold spread: 0.0014
-> the ordering is within noise; prefer the simpler candidate


,model.nb.alpha,model.vectorizer.max_features,split.min_path_count,model.vectorizer.ngram_range,model.vectorizer.min_df,model.vectorizer.max_df,model.vectorizer.stop_words,model.vectorizer.kind,model.vectorizer.sublinear_tf,model.vectorizer.norm,model.vectorizer.use_idf,model.nb.fit_prior,model.prediction_mode,model.vectorizer.unordered_bigrams,n_rows,n_paths,min_path_count_applied,path_accuracy,path_accuracy_std,fit_seconds,correct_through_Level 1 ID,correct_through_Level 2 ID,correct_through_Level 3 ID,correct_through_Level 4 ID
0,0.001,200000,20,"(1, 2)",2,0.95,english,tfidf,True,l2,False,True,joint_path,False,284708,1658,20,0.733074,0.001406,216.337324,0.907653,0.878486,0.824571,0.733074
1,0.001,150000,20,"(1, 2)",2,0.95,english,tfidf,True,l2,False,True,joint_path,False,284708,1658,20,0.733074,0.001406,223.203180,0.907653,0.878486,0.824571,0.733074
2,0.001,100000,20,"(1, 2)",2,0.95,english,tfidf,True,l2,False,True,joint_path,False,284708,1658,20,0.733074,0.001406,208.359770,0.907653,0.878486,0.824571,0.733074


### Does each parameter matter at all?

The marginal effect of one parameter, averaged over every setting of the
others. More trustworthy than the single best row, and it tells you what
to drop from the next pass.

In [0]:
for _param in GRID:
    print(f"\n{_param}")
    print(summarise_by(results, _param).to_string(index=False))


model.nb.alpha
 model.nb.alpha  mean_accuracy  best_accuracy  n_candidates  mean_fit_seconds
          0.001       0.733074       0.733074             3        215.966758

model.vectorizer.max_features
 model.vectorizer.max_features  mean_accuracy  best_accuracy  n_candidates  mean_fit_seconds
                        100000       0.733074       0.733074             1        208.359770
                        150000       0.733074       0.733074             1        223.203180
                        200000       0.733074       0.733074             1        216.337324

split.min_path_count
 split.min_path_count  mean_accuracy  best_accuracy  n_candidates  mean_fit_seconds
                   20       0.733074       0.733074             3        215.966758

model.vectorizer.ngram_range
model.vectorizer.ngram_range  mean_accuracy  best_accuracy  n_candidates  mean_fit_seconds
                      (1, 2)       0.733074       0.733074             3        215.966758

model.vectorizer.min_d

---
## Save Optimal Parameters

`best_params` returns the winning combination in its original form, so
tuples come back as tuples rather than the strings used for display.

In [0]:
params = best_params(results)
params

{'model.nb.alpha': 0.001,
 'model.vectorizer.max_features': 200000,
 'split.min_path_count': 20,
 'model.vectorizer.ngram_range': (1, 2),
 'model.vectorizer.min_df': 2,
 'model.vectorizer.max_df': 0.95,
 'model.vectorizer.stop_words': 'english',
 'model.vectorizer.kind': 'tfidf',
 'model.vectorizer.sublinear_tf': True,
 'model.vectorizer.norm': 'l2',
 'model.vectorizer.use_idf': False,
 'model.nb.fit_prior': True,
 'model.prediction_mode': 'joint_path',
 'model.vectorizer.unordered_bigrams': False}

In [0]:
cfg_best = apply_params(cfg, params)
cfg_best.model

ModelConfig(n_levels=4, prediction_mode='joint_path', normalise_probabilities=None, batch_size=1000, fit_chunk_size=20000, dtype='float32', vectorizer=VectorizerConfig(kind='tfidf', max_features=200000, ngram_range=(1, 2), min_df=2, max_df=0.95, stop_words='english', lowercase=True, binary=False, sublinear_tf=True, norm='l2', use_idf=False, unordered_bigrams=False, extra={}), nb=NBConfig(alpha=0.001, fit_prior=True, class_prior=None))

The JSON record carries the settings **and** how they were found —
candidates searched, folds, rows, and the accuracy achieved. A bare dict
of settings invites "searched over what?", and a file that cannot answer
gets distrusted and re-derived.

`param_types` is there because JSON has no tuples: without it
`ngram_range` returns as `[1, 2]` and sklearn is handed a list where it
expects a pair.

In [0]:
PARAMS_PATH = "../outputs/best_params.json"

record = save_params(
    results, PARAMS_PATH,
    notes=f"sample_frac={SAMPLE_FRAC}, n_folds={N_FOLDS}",
)
print(f"wrote {PARAMS_PATH}\n")
print(json.dumps(record, indent=2))

wrote ../outputs/best_params.json

{
  "created_utc": "2026-09-15T00:31:14+00:00",
  "params": {
    "model.nb.alpha": 0.001,
    "model.vectorizer.max_features": 200000,
    "split.min_path_count": 20,
    "model.vectorizer.ngram_range": [
      1,
      2
    ],
    "model.vectorizer.min_df": 2,
    "model.vectorizer.max_df": 0.95,
    "model.vectorizer.stop_words": "english",
    "model.vectorizer.kind": "tfidf",
    "model.vectorizer.sublinear_tf": true,
    "model.vectorizer.norm": "l2",
    "model.vectorizer.use_idf": false,
    "model.nb.fit_prior": true,
    "model.prediction_mode": "joint_path",
    "model.vectorizer.unordered_bigrams": false
  },
  "param_types": {
    "model.nb.alpha": "float",
    "model.vectorizer.max_features": "int",
    "split.min_path_count": "int",
    "model.vectorizer.ngram_range": "tuple",
    "model.vectorizer.min_df": "int",
    "model.vectorizer.max_df": "float",
    "model.vectorizer.stop_words": "str",
    "model.vectorizer.kind": "str",
    "

In [0]:
# Read it back the way notebooks 3 and 4 will, and confirm the round-trip.
_loaded = load_params(PARAMS_PATH)
assert _loaded["params"] == params, "round-trip changed the parameters"
print("round-trip intact:", _loaded["params"])

round-trip intact: {'model.nb.alpha': 0.001, 'model.vectorizer.max_features': 200000, 'split.min_path_count': 20, 'model.vectorizer.ngram_range': (1, 2), 'model.vectorizer.min_df': 2, 'model.vectorizer.max_df': 0.95, 'model.vectorizer.stop_words': 'english', 'model.vectorizer.kind': 'tfidf', 'model.vectorizer.sublinear_tf': True, 'model.vectorizer.norm': 'l2', 'model.vectorizer.use_idf': False, 'model.nb.fit_prior': True, 'model.prediction_mode': 'joint_path', 'model.vectorizer.unordered_bigrams': False}


### Confirm on everything

The search above ran on a sample with few folds — right for exploring,
too optimistic to quote. Re-run just the winner at full size before
adopting it, and overwrite the record with those numbers.

In [0]:
# confirmed = grid_search(
#     cfg, df, {k: [v] for k, v in params.items()},
#     n_folds=5, sample_frac=None,
# )
# save_params(confirmed, PARAMS_PATH, notes="confirmed at full size, 5 folds")
# confirmed

In [0]:
# The full results table, for the record.
RESULTS_PATH = "../outputs/grid_search_results.csv"

results.to_csv(RESULTS_PATH, index=False)
print("wrote", RESULTS_PATH)

wrote ../outputs/grid_search_results.csv
